# Chapter 05-03 · Multiple regression, and what a coefficient means

**Label:** Core  |  **Time:** ~60 minutes  |  **Difficulty:** the arithmetic is easy, the interpretation
is not

**Prerequisites:** 05-02 for the one-feature fit, 03-07 for `X @ w`, 02-06 for Simpson's paradox.

**Position in the learning path:** module 05, chapter 3 of 12.

---

## Why this matters

Adding a second feature to a regression is trivially easy: one more column, one more coefficient, the same
`fit` call. **Reading the result is not easy at all**, and this is the chapter where most
misunderstandings of regression begin.

Here is the thing to be ready for. In the data below, fitting price on age alone gives a coefficient of
**-1.03**: older houses are cheaper. Adding one more column - the size of the house - changes it to
**+0.82**: older houses are *dearer*.

**Both numbers are correct.** Neither is a bug, neither is a better fit of the same quantity, and they
answer different questions. Knowing which question you asked is the skill this chapter is about.

## What you will be able to do

- Fit and read a regression with several features
- State precisely what "holding the others constant" means - and demonstrate it mechanically
- Predict when adding a feature will change another's coefficient, and by how much
- Recognise collinearity, and say what it damages and what it leaves alone
- Decide when a coefficient may be interpreted and when only the prediction may be used

## Warm-up: retrieve, do not reread

1. What does `X @ w` compute, and what shape is the result?
2. In 05-02, what does the slope's denominator, `sum((x - x̄)²)`, do?
3. In 02-06, what was Simpson's paradox?

<br>

*Answers: (1) a weighted sum per row - one prediction per row, shape `(n,)`. (2) converts the numerator
into y-units per x-unit. (3) a relationship that reverses when the data is split by a third variable.*

## Three hundred houses

Synthetic, so the truth is known and every claim can be checked against it.

**How it was built:** price rises by **3.0** per square metre and by **0.9** per year of age. Older houses
also happen to be *smaller* in this town - the older stock is cottages, the new build is large.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# SYNTHETIC: 300 houses. TRUTH: price = 40 + 3.0 x size + 0.9 x age + noise,
# and older houses are systematically smaller.
rng = np.random.default_rng(53)
n_houses = 300
age = rng.uniform(0, 80, n_houses)
size = 130 - 0.7 * age + rng.normal(0, 10, n_houses)
price = 40 + 3.0 * size + 0.9 * age + rng.normal(0, 20, n_houses)

houses = pd.DataFrame({"size_m2": size, "age_years": age, "price_k": price})

print("300 houses")
print("  correlation between size and age : %+.4f" % houses.size_m2.corr(houses.age_years))
print()
print("the truth, which we get to see because the data is synthetic:")
print("  each extra square metre adds  3.0 thousand")
print("  each extra year of age adds   0.9 thousand")

## The sign flip

Three fits: price on size alone, price on age alone, and price on both.

**Predict before running:** what will the age coefficient be in each?

In [ ]:
rows = []
for columns in [["size_m2"], ["age_years"], ["size_m2", "age_years"]]:
    fitted = LinearRegression().fit(houses[columns], houses.price_k)
    entry = {"features used": " + ".join(columns), "size_m2": None, "age_years": None}
    for name, value in zip(columns, fitted.coef_):
        entry[name] = round(float(value), 4)
    rows.append(entry)
rows.append({"features used": "THE TRUTH", "size_m2": 3.0, "age_years": 0.9})
print(pd.DataFrame(rows).to_string(index=False, na_rep="-"))

**Age alone: -1.0304. Age alongside size: +0.8213.** The sign reverses, and the second number is close to
the truth of +0.9.

Size behaves the same way in miniature: **+1.92 alone against +2.80 with age included**, where the truth
is 3.0. The single-feature fits are not slightly off - they are answering a different question.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
labels = ["size, alone", "size, with age", "age, alone", "age, with size"]
values, truths = [1.9242, 2.7977, -1.0304, 0.8213], [3.0, 3.0, 0.9, 0.9]
positions = np.arange(4)
ax.bar(positions, values, width=0.55, color=["#7db4e6", "#0072B2", "#f0a882", "#D55E00"])
for position, (value, truth) in enumerate(zip(values, truths)):
    ax.plot([position - 0.32, position + 0.32], [truth, truth], color="#000000", linewidth=2.4)
    inside = value / 2 if value > 0 else value / 2
    ax.text(position, inside, "%+.2f" % value, ha="center", va="center",
            fontsize=11, fontweight="bold", color="white")
ax.plot([], [], color="#000000", linewidth=2.4, label="the true effect")
ax.axhline(0, color="#666666", linewidth=1)
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=9.5)
ax.set_ylabel("fitted coefficient")
ax.set_ylim(-1.7, 3.7)
ax.set_title("Alone, both coefficients are wrong. Together, both are close to the truth", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### The two questions

- **"How does price differ between older and newer houses?"** Answer: older houses are cheaper, by about
  1.03 thousand per year. **True, and useful if you are describing the housing stock.**
- **"How does price differ between two houses of the same size, one older than the other?"** Answer: the
  older one is dearer, by about 0.82 thousand per year. **True, and useful if you are valuing a
  particular house.**

The first question mixes age with everything age is associated with - and in this town, older means
smaller, and smaller means cheaper. That size effect is larger than the age effect and it is pulling in
the opposite direction, so it wins.

**A single-feature coefficient contains every path from the feature to the target.** A multiple-regression
coefficient contains only the direct one, given the columns present.

Here it is drawn.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 5)) 

overall = LinearRegression().fit(houses[["age_years"]], houses.price_k)
grid = np.linspace(0, 80, 2)
left.scatter(houses.age_years, houses.price_k, s=16, color="#999999", alpha=0.6)
left.plot(grid, overall.intercept_ + overall.coef_[0] * grid, color="#D55E00", linewidth=2.6,
          label="all houses: %+.2f per year" % overall.coef_[0])
left.set_xlabel("age (years)")
left.set_ylabel("price (thousands)")
left.set_title("Ignoring size: older is cheaper", fontsize=11.5)
left.legend(fontsize=9)

bands = pd.qcut(houses.size_m2, 4, labels=False)
colours = ["#cfe3f3", "#7db4e6", "#0072B2", "#004c7a"]
for band in range(4):
    inside = bands == band
    subset = houses[inside]
    within = LinearRegression().fit(subset[["age_years"]], subset.price_k)
    span = np.linspace(subset.age_years.min(), subset.age_years.max(), 2)
    right.scatter(subset.age_years, subset.price_k, s=16, color=colours[band], alpha=0.85)
    right.plot(span, within.intercept_ + within.coef_[0] * span, color=colours[band], linewidth=2.6)
right.plot([], [], color="#0072B2", linewidth=2.6, label="within each quarter of size")
right.set_xlabel("age (years)")
right.set_ylabel("price (thousands)")
right.set_title("Split by size: within every band, older is dearer", fontsize=11.5)
right.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("slope of price on age, within each quarter of the size distribution:")
for band in range(4):
    subset = houses[bands == band]
    within = LinearRegression().fit(subset[["age_years"]], subset.price_k)
    print("  size band %d (%3.0f to %3.0f m2): %+.4f per year"
          % (band + 1, subset.size_m2.min(), subset.size_m2.max(), within.coef_[0]))

**Every band slopes upward; the cloud as a whole slopes downward.** That is 02-06's Simpson's paradox,
and multiple regression is the tool that resolves it.

The within-band slopes run from **+0.28 to +0.65** - all positive, all below the regression's +0.82. They
are *not* the coefficient, and the gap is instructive: houses inside a band still differ in size by up to
35 m², and that leftover size variation still drags the within-band slope down. **Banding is an
approximation to holding size constant.** The next section does it exactly.

The picture is nonetheless the right mental image: **stay inside one colour and compare houses that differ
in age but not much in size.**

## "Holding the others constant", done mechanically

The size bands are an approximation - houses within a band still differ in size. There is an exact version,
and it is the best explanation of what a regression coefficient is.

**Take the part of age that size cannot explain, and the part of price that size cannot explain, and fit a
line between those two leftovers.**

In [ ]:
both = LinearRegression().fit(houses[["size_m2", "age_years"]], houses.price_k)

# strip out everything size can account for, from age and from price separately
age_given_size = houses.age_years - LinearRegression().fit(
    houses[["size_m2"]], houses.age_years).predict(houses[["size_m2"]])
price_given_size = houses.price_k - LinearRegression().fit(
    houses[["size_m2"]], houses.price_k).predict(houses[["size_m2"]])

partial = LinearRegression().fit(age_given_size.to_numpy().reshape(-1, 1), price_given_size)

print("coefficient on age in the two-feature regression : %.6f" % both.coef_[1])
print("slope of leftover price on leftover age          : %.6f" % partial.coef_[0])
print("identical:", bool(np.isclose(both.coef_[1], partial.coef_[0])))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))

axes[0].scatter(houses.size_m2, houses.age_years, s=14, color="#999999", alpha=0.6)
size_to_age = LinearRegression().fit(houses[["size_m2"]], houses.age_years)
span = np.linspace(houses.size_m2.min(), houses.size_m2.max(), 2)
axes[0].plot(span, size_to_age.intercept_ + size_to_age.coef_[0] * span, color="#D55E00",
             linewidth=2.4)
axes[0].set_xlabel("size (m2)")
axes[0].set_ylabel("age (years)")
axes[0].set_title("1. what size already tells us about age", fontsize=10.5)

axes[1].scatter(houses.size_m2, price_given_size, s=14, color="#999999", alpha=0.6)
axes[1].axhline(0, color="#D55E00", linewidth=2.4)
axes[1].set_xlabel("size (m2)")
axes[1].set_ylabel("price, with size removed")
axes[1].set_title("2. price with size stripped out", fontsize=10.5)

axes[2].scatter(age_given_size, price_given_size, s=14, color="#0072B2", alpha=0.7)
leftover_span = np.linspace(age_given_size.min(), age_given_size.max(), 2)
axes[2].plot(leftover_span, partial.intercept_ + partial.coef_[0] * leftover_span,
             color="#009E73", linewidth=2.8)
axes[2].set_xlabel("age, with size removed")
axes[2].set_ylabel("price, with size removed")
axes[2].set_title("3. slope here = %.4f = the age coefficient" % partial.coef_[0], fontsize=10.5)

plt.tight_layout()
plt.show()

**The third panel's slope is the age coefficient, to six decimal places.** This is the
Frisch-Waugh-Lovell result, and it turns a phrase into a procedure:

> **"The coefficient on age, holding size constant" means: remove everything size can explain - from age
> and from price - and fit a line to what is left.**

Three consequences follow immediately, and they are the practical content of this chapter.

**1. A coefficient depends on which other columns are in the model.** It is not a property of the feature;
it is a property of the feature *given the others*. Add a column, and every other coefficient can change.
Adding a column that is unrelated to the rest changes nothing; adding a correlated one changes everything.

**2. "Holding constant" is arithmetic, not an experiment.** Nothing was held constant in the world - no
house was rebuilt at a different age. The regression compared houses that *happened* to differ in age
while being similar in size, and if your data contains no such comparisons, the coefficient is an
extrapolation. **This is why "we controlled for X" is a claim about the data, not a guarantee.**

**3. The leftover age has less spread than the original.** That third panel is narrower than the raw age
axis, because size already explained most of it. **The more the two features overlap, the less is left to
estimate the coefficient from** - which is the next section.

In [ ]:
print("spread of age as recorded          : %.3f years" % houses.age_years.std())
print("spread of age once size is removed : %.3f years" % age_given_size.std())
print("so %.0f%% of the variation in age was already predictable from size"
      % (100 * (1 - age_given_size.var() / houses.age_years.var())))

## Collinearity: when the features overlap

The size-age correlation here is **-0.84**, which is high. What does that cost?

The honest way to find out is to refit on resampled data and watch the coefficients move. Three pairs of
features, engineered to be independent, correlated, and nearly identical.

**Predict before running:** what happens to the coefficients, and what happens to the predictions?

In [ ]:
def instability(correlation, draws=300, rows=300, seed=1):
    generator = np.random.default_rng(seed)
    first = generator.normal(size=rows)
    second = correlation * first + np.sqrt(max(1e-9, 1 - correlation ** 2)) * generator.normal(size=rows)
    target = 2 * first + 2 * second + generator.normal(0, 1, rows)
    design = np.column_stack([first, second])

    coefficients, predictions = [], []
    for _ in range(draws):
        picked = generator.integers(0, rows, rows)
        refit = LinearRegression().fit(design[picked], target[picked])
        coefficients.append(refit.coef_)
        predictions.append(refit.predict(design[:20]))
    coefficients, predictions = np.array(coefficients), np.array(predictions)
    return {"correlation": round(float(np.corrcoef(first, second)[0, 1]), 3),
            "sd of coefficient 1": round(float(coefficients[:, 0].std()), 4),
            "sd of coefficient 2": round(float(coefficients[:, 1].std()), 4),
            "sd of the predictions": round(float(predictions.std(axis=0).mean()), 4),
            "coefficients": coefficients}


results = [instability(c) for c in [0.0, 0.95, 0.999]]
print(pd.DataFrame([{k: v for k, v in r.items() if k != "coefficients"} for r in results]).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4), sharex=True, sharey=True)
for ax, result, label in zip(axes, results, ["independent", "correlated", "nearly identical"]):
    ax.scatter(result["coefficients"][:, 0], result["coefficients"][:, 1], s=12, alpha=0.5,
               color="#0072B2")
    ax.plot([2], [2], "*", color="#D55E00", markersize=18)
    ax.set_xlabel("coefficient on feature 1")
    ax.set_title("%s (r = %.2f)" % (label, result["correlation"]), fontsize=11)
axes[0].set_ylabel("coefficient on feature 2")
axes[0].set_xlim(-2.5, 6.5)
axes[0].set_ylim(-2.5, 6.5)
fig.suptitle("300 refits each. The star is the truth (2, 2)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

**The coefficient spread goes from 0.056 to 1.263 - twenty-three times wider - while the prediction
spread does not move at all: 0.0802 in all three.**

That exact equality is a property of how the three datasets were built, and it is worth being explicit
about because it is the mechanism rather than a fluke. The second feature is always
`correlation × first + something × e`, built from the **same** two underlying draws - so
`span{first, second}` is the *same plane* for every correlation. Only the **basis** of that plane changes.
The model can therefore reach exactly the same set of predictions in all three cases, and its residuals
are the identical vector (standard deviation 1.0528 throughout). **Collinearity rotated the coordinates
without moving the space.**

The third panel is the picture to remember. The cloud is not a blob, it is a **long thin diagonal**: when
one coefficient is over-estimated the other is under-estimated by nearly the same amount. The model cannot
tell the two features apart, so it can only pin down their **sum**, and it distributes that sum between
them almost arbitrarily.

That gives collinearity a precise description:

| What collinearity damages | What it leaves alone |
|---|---|
| the **individual coefficients** - unstable, huge, sometimes wrong-signed | the **predictions** - as accurate as ever |
| any statement of the form "this feature is worth X" | any statement of the form "this model predicts Y" |
| feature importance read off the coefficients | the fit, the residuals, R-squared |

**So collinearity is not a prediction problem, it is an interpretation problem** - and whether it matters
depends entirely on which of the two you came for. A model built to forecast can ignore it. A model built
to tell somebody how much an extra bedroom is worth cannot.

## The same thing on real data

The houses above were synthetic so the truth could be checked. Here is the effect on data nobody
engineered: **California housing**, 02-08's census districts, which contains two columns that are almost
the same measurement - the average number of rooms per household and the average number of bedrooms.

In [ ]:
from sklearn.datasets import fetch_california_housing

districts = fetch_california_housing(as_frame=True).frame.sample(3000, random_state=0)
print("correlation between AveRooms and AveBedrms : %.4f"
      % districts.AveRooms.corr(districts.AveBedrms))
print()

rows = []
for columns in [["AveRooms"], ["AveBedrms"], ["AveRooms", "AveBedrms"]]:
    fitted = LinearRegression().fit(districts[columns], districts.MedHouseVal)
    entry = {"features used": " + ".join(columns), "AveRooms": None, "AveBedrms": None}
    for name, value in zip(columns, fitted.coef_):
        entry[name] = round(float(value), 4)
    rows.append(entry)
print(pd.DataFrame(rows).to_string(index=False, na_rep="-"))

Read down the columns.

**Alone, more rooms is worth +0.0735** and **more bedrooms is worth -0.1730.** Together, more rooms
becomes **+0.2901** - four times larger - and more bedrooms becomes **-1.4405**, more than eight times
larger.

Neither number is a mistake, and the pair has a coherent reading: **holding the number of bedrooms fixed,
extra rooms mean a bigger house and cost more; holding total rooms fixed, having more of them be bedrooms
means smaller rooms and a cheaper district.** That is a real and interesting statement, and it is only
visible in the two-feature model.

But notice how large the coefficients became. **That growth is the signature of collinearity**: the two
features are 0.81 correlated, so the model is estimating a difference between two nearly-identical
quantities, and small differences between large numbers are exactly what is hard to pin down. A district
with an unusual rooms-to-bedrooms ratio now moves the prediction a long way.

**A quick check worth running whenever coefficients look surprisingly large.**

In [ ]:
def variance_inflation(frame, columns):
    scores = []
    for target_column in columns:
        others = [c for c in columns if c != target_column]
        explained = LinearRegression().fit(frame[others], frame[target_column]).score(
            frame[others], frame[target_column])
        scores.append({"column": target_column,
                       "R2 from the other columns": round(explained, 4),
                       "variance inflation factor": round(1 / (1 - explained), 2)})
    return pd.DataFrame(scores)


print(variance_inflation(districts, ["AveRooms", "AveBedrms", "MedInc", "HouseAge"]).to_string(index=False))

In [ ]:
inflation = variance_inflation(districts, ["AveRooms", "AveBedrms", "MedInc", "HouseAge"])

fig, ax = plt.subplots(figsize=(8.5, 3.8))
colours = ["#D55E00" if v >= 5 else "#e8a33d" if v >= 2 else "#0072B2"
           for v in inflation["variance inflation factor"]]
ax.barh(range(len(inflation)), inflation["variance inflation factor"], color=colours, height=0.6)
for position, value in enumerate(inflation["variance inflation factor"]):
    ax.text(value + 0.12, position, "%.2f  (%.1fx noisier)" % (value, np.sqrt(value)),
            va="center", fontsize=9.5)
ax.axvline(1, color="#000000", linewidth=1.2)
ax.set_yticks(range(len(inflation)))
ax.set_yticklabels(inflation["column"], fontsize=10)
ax.set_xlim(0, 9)
ax.set_xlabel("variance inflation factor")
ax.set_title("How much each coefficient's uncertainty is inflated by the others", fontsize=11.5)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**The variance inflation factor is `1 / (1 - R²)`, where the R-squared is from predicting that column
using the others.** It says how many times wider the coefficient's uncertainty is than it would be if the
column were unrelated to the rest.

`AveRooms` sits at **6.01** and `AveBedrms` at **5.25**, meaning their coefficients are roughly **2.4
times noisier** (the square root of six) than they would be if the columns were independent. `MedInc` is
at 2.06 and `HouseAge` at 1.04, essentially unaffected.

Note that both inflation factors are higher than the plain correlation of 0.81 would suggest, because a
VIF accounts for *all* the other columns at once - `AveRooms` is predictable not only from `AveBedrms` but
partly from `MedInc` too.

Rules of thumb put "concerning" somewhere between 5 and 10, but **the number is a description rather than
a verdict**. What to do about it depends on why you built the model:

| If you want | Do |
|---|---|
| predictions | nothing. Collinearity does not hurt them |
| to interpret coefficients | drop one, combine them (`rooms - bedrooms`, or the ratio), or collect data that separates them |
| both | fit two models and say so - one for predicting, one for explaining |

The third row is the honest answer more often than people expect. **"Which feature matters most" and
"what will the price be" are different questions, and one model does not have to answer both.**

## Reading a multiple regression, in order

The checklist this chapter arrives at:

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.2))
steps = [
    ("1  What is in the model?", "#cfe3f3",
     "a coefficient means 'given these other columns'. List them before reading anything"),
    ("2  Are the features correlated?", "#cfe3f3",
     "check the correlations and the inflation factors. High means the split is arbitrary"),
    ("3  What are the units?", "#f6d3bd",
     "a coefficient without units is a rumour - 03-06. Compare across features only when scaled"),
    ("4  Is zero inside the data?", "#f6d3bd",
     "if not, the intercept is an extrapolation. Centring fixes it - 05-02's E18"),
    ("5  Am I interpreting or predicting?", "#cfe8dc",
     "collinearity ruins the first and leaves the second untouched"),
]
for position, (title, colour, detail) in enumerate(steps):
    y = len(steps) - position - 1
    ax.add_patch(plt.Rectangle((0.05, y + 0.08), 3.3, 0.84, facecolor=colour, edgecolor="white",
                               linewidth=2.5))
    ax.text(1.7, y + 0.5, title, ha="center", va="center", fontsize=11.5, fontweight="bold")
    ax.text(3.55, y + 0.5, detail, va="center", fontsize=9.5, color="#333333")
ax.set_xlim(0, 12.4)
ax.set_ylim(-0.15, len(steps) + 0.35)
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Before reading any coefficient", fontsize=13)
plt.tight_layout()
plt.show()

## Common misconceptions

**"The coefficient tells you the effect of the feature."**
It tells you the effect *given the other columns in the model*, estimated from whatever comparisons the
data happens to contain. Change the columns and it changes.

**"A sign flip means something went wrong."**
Both signs here are correct answers to different questions. The flip is information about the data, not an
error.

**"Adding features makes coefficients more accurate."**
Adding a *relevant* one can remove bias. Adding a *correlated* one inflates variance. Usually both happen
at once, which is 05-09's subject.

**"Collinearity ruins the model."**
It ruins the coefficients. The predictions were fine throughout - 0.083 against 0.085 in this chapter's
experiment.

**"A large coefficient means an important feature."**
It may mean a small-scaled feature, or two collinear features cancelling. Compare magnitudes only after
standardising, and even then read them as "per standard deviation", not as importance.

**"'We controlled for X' means the comparison is clean."**
It means X was included as a linear term. If the relationship is not linear, or if the data contains no
rows that differ in one feature while matching on the other, the control is nominal.

## Exercises

Solutions: `solutions/05_regression/05-03_multiple_linear_solutions.ipynb`.

### Quick understanding

**E1.** In one sentence, what does a multiple-regression coefficient mean?

**E2.** Explain the Frisch-Waugh-Lovell procedure in three steps.

**E3.** What does collinearity damage, and what does it leave alone?

### Hand calculation

**E4.** A regression of price on size alone gives +1.92. The true direct effect of size is +3.0. Age has a
true effect of +0.9 and the slope of size on age is -0.66. Use the omitted-variable formula
`marginal = direct + (effect of the omitted) × (slope of the omitted on the included)` to predict the
*age-alone* coefficient, and compare it with the -1.03 that was measured.

**E5.** Two features have a correlation of 0.9. Compute the variance inflation factor. How many times
wider is the coefficient's standard error than if they were independent?

**E6.** A model has `price = 40 + 3.0 × size_m2`. Rewrite it with size in **square feet**
(1 m² = 10.764 sq ft). Give the new coefficient, and say whether the model changed.

**E7.** A standardised regression reports coefficients of 0.62 for income and 0.11 for age. Say what each
means in words, and what you would need to convert them back into currency and years.

### Coding

**E8.** Write `partial_coefficient(frame, target, feature, controls)` implementing Frisch-Waugh-Lovell,
and confirm it reproduces every coefficient of the two-feature house model.

**E9.** Add a **pure noise** column to the house data and refit. How much do the size and age coefficients
move? Repeat with 20 noise columns and report the same numbers.

**E10.** Build the collinearity experiment yourself for correlations 0, 0.5, 0.9, 0.99 and 0.999. Plot
coefficient spread and prediction spread against correlation on the same axes, with a log y-scale.

**E11.** On the California data, replace `AveRooms` and `AveBedrms` with two engineered columns:
`AveRooms` and `AveBedrms / AveRooms`. Report the new coefficients and inflation factors, and say what each
coefficient now means.

**E12.** Fit the house model with `size_m2` in metres squared and again with it standardised. Confirm the
predictions are identical and the coefficients are not, and state the exact conversion between them.

### Interpretation

**E13.** A colleague's model of salary includes both `years_experience` and `age`, and reports a
**negative** coefficient on age. Explain how that can be correct, and what it means.

**E14.** A hospital model predicting length of stay includes a coefficient of -1.3 days on "admitted via
the emergency department". A manager proposes closing the emergency route to shorten stays. Respond.

### Debugging

**E15.** Adding one feature changes another coefficient from +2.1 to -0.4. Give the two things you would
check, in order.

**E16.** A model's coefficients are enormous - some in the millions - and its predictions are fine. Name
the cause and two fixes.

### Exam and interview reasoning

**E17.** "What does a regression coefficient mean?" Answer in under a minute, then handle: "so it tells us
what would happen if we changed that variable?"

### Transfer to a different situation

**E18.** You are modelling crop yield from rainfall, fertiliser and soil nitrogen, where fertiliser is
applied *in response to* measured nitrogen. Say what the fertiliser coefficient will mean, and what you
would need in order to interpret it as an effect.

### Explain it to someone non-technical

**E19.** Explain in under 90 words how the same data can say older houses are cheaper *and* older houses
are dearer, without either being wrong.

### Optional challenge

**E20.** Demonstrate that a multiple regression's coefficients can be obtained one at a time by repeated
partialling out: fit the three-feature model on the house data plus a noise column, then reproduce each
coefficient using Frisch-Waugh-Lovell against the other two. Then show what happens when two of the
features are *perfectly* collinear - what does the library return, and why is that answer as good as any
other?

In [ ]:
# Your workspace. In memory: houses, age_given_size, price_given_size, both, partial,
# instability, results, districts, variance_inflation.

## Mastery check

- [ ] Fit and read a regression with several features
- [ ] State what a coefficient means, including the phrase "given the other columns"
- [ ] Perform Frisch-Waugh-Lovell and confirm it reproduces a coefficient
- [ ] Predict the direction of an omitted-variable shift before fitting
- [ ] Compute a variance inflation factor and say what it does and does not imply
- [ ] Decide whether a model is for interpreting or predicting, and say what follows

## What should now feel instinctive

- Listing the other columns before reading any coefficient
- Checking feature correlations before believing a coefficient's size
- Reading a sign flip as information rather than as a bug
- Refusing to convert a coefficient into a recommended action without an argument about causation
- Separating "which feature matters" from "what will the number be"

## Flashcards

| Front | Back |
|---|---|
| A multiple-regression coefficient | The effect of that feature **given the other columns**, from the comparisons the data contains |
| Frisch-Waugh-Lovell | Remove the controls from the feature and from the target; fit a line to the leftovers |
| The sign flip here | Age alone -1.03, age given size +0.82. Both correct, different questions |
| Omitted-variable formula | `marginal = direct + (omitted effect) × (slope of omitted on included)` |
| Collinearity damages | Individual coefficients: spread went 0.056 to 1.214 at r = 0.999 |
| Collinearity leaves alone | Predictions: spread 0.083 against 0.085 |
| Why the coefficient cloud is diagonal | The model can pin down the sum but not the split |
| Variance inflation factor | `1 / (1 - R² from the other columns)`. AveRooms and AveBedrms sit near 3 |
| Interpreting versus predicting | Collinearity ruins the first only. Two models is a legitimate answer |
| "We controlled for X" | A claim about the data and the model's shape, not a guarantee |

## Next

**05-04 · Metrics: MAE, MSE, RMSE, MAPE's trouble, R-squared.** This chapter and the two before it have
quietly used four different error measures and one skill score. The next one takes them properly: what
each is in the units of the problem, which are sensitive to outliers, why MAPE misbehaves - 05-01 showed
it asking for a constant below the median - and what R-squared does and does not tell you.